<a href="https://colab.research.google.com/github/rigayus24ti-cmd/Distribusi-Probabilitas/blob/main/KUESIONER_PENGGUNAAN_AI_DAN_KEBIASAAN_PENGGUNAAN_TEKNOLOGI_DIGITAL_MAHASISWA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files
from IPython.display import display



In [ ]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses 3 TIC).xlsx to Blank Quiz (Responses 3 TIC).xlsx
File terbaca: Blank Quiz (Responses 3 TIC).xlsx


In [ ]:
JUMLAH_KOLOM_DEMOGRAFI = 5  # A-E: Timestamp, Kelas, Jenis Kelamin, Jumlah Platform, Jumlah Premium

df = pd.read_excel(nama_file)

print("Jumlah responden :", len(df))
print("Jumlah kolom     :", df.shape[1])

Jumlah responden : 40
Jumlah kolom     : 35


In [ ]:
kolom_likert = [
    c for c in df.columns[JUMLAH_KOLOM_DEMOGRAFI:]
    if "[" in str(c)
]

butir = df[kolom_likert].copy()

print("Jumlah butir Likert :", butir.shape[1])
print("Data kosong         :", int(butir.isna().sum().sum()))

Jumlah butir Likert : 29
Data kosong         : 0


In [ ]:
skala = {
    "Sangat Tidak Setuju (STJ)": 1,
    "Sangat Tidak Setuju (STS)": 1,  # cadangan jika ada variasi label
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

In [ ]:
butir_clean = butir.apply(lambda col: col.astype(str).str.strip())

num = butir_clean.replace(skala).apply(pd.to_numeric, errors="coerce")

gagal = int(num.isna().sum().sum())
print("Sel yang gagal dikonversi:", gagal)

if gagal > 0:
    print("\nKolom yang masih mengandung nilai tidak terbaca:")
    print(num.isna().sum()[num.isna().sum() > 0])
    raise ValueError(
        "Ada jawaban yang tidak cocok dengan kamus skala. "
        "Periksa label jawaban pada data."
    )

display(num.head())

Sel yang gagal dikonversi: 0


/tmp/ipykernel_614/917187404.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir_clean.replace(skala).apply(pd.to_numeric, errors="coerce")


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Respon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Respon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Respon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Respon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Respon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,5,5,4,4,4,4,4,5,4,4,...,2,5,4,5,4,5,5,4,4,5
1,4,4,4,4,4,4,4,5,4,4,...,4,2,4,4,4,5,4,5,5,5
2,5,5,4,4,4,4,4,4,4,4,...,2,4,5,4,2,5,5,5,4,5
3,5,4,5,2,4,5,5,4,4,2,...,2,4,4,4,4,5,4,4,4,4
4,5,5,4,4,5,5,5,4,2,4,...,2,2,2,2,4,5,5,4,5,5


In [ ]:
dimensi = {}

for kolom in num.columns:
    match = re.match(r"^(.+?)\s*\[", str(kolom))
    if match is None:
        print("Kolom tidak cocok format dimensi:", kolom)
        continue

    nama_dimensi = match.group(1).strip()
    dimensi.setdefault(nama_dimensi, []).append(kolom)

print("\nDimensi dan jumlah butir:")
for nama, daftar in dimensi.items():
    print(f"- {nama}: {len(daftar)} butir")


Dimensi dan jumlah butir:
- Penggunaan AI pada tugas atau projek kuliah: 7 butir
- Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
- Kebiasaan penggunaan teknologi digital: 6 butir
- Prokrastinasi akademik.: 4 butir
- Respon emosional: 5 butir


In [ ]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)

print(f"\nn = {n} | df = {n-2} | r tabel (α=0,05; dua sisi) = {r_tabel:.3f}")

hasil_validitas = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])

        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)

print("\n=== TABEL VALIDITAS ===")
display(tabel_validitas)



n = 40 | df = 38 | r tabel (α=0,05; dua sisi) = 0.312

=== TABEL VALIDITAS ===


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.419,0.222,0.0071,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.390,0.224,0.0129,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.626,0.418,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.506,0.205,0.0009,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.793,0.597,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.634,0.424,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.231,-0.061,0.1510,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.578,0.437,0.0001,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.708,0.541,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.804,0.651,0.0000,Valid


In [ ]:
def cronbach_alpha(x):
    x = x.dropna()
    k = x.shape[1]
    if k < 2:
        return np.nan
    return k / (k - 1) * (
        1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1)
    )

def kategori_alpha(a):
    if pd.isna(a):
        return "Tidak dapat dihitung"
    if a >= 0.90:
        return "Sangat baik"
    elif a >= 0.80:
        return "Baik"
    elif a >= 0.70:
        return "Reliabel"
    else:
        return "Tidak reliabel"

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)

    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori_alpha(a),
    })

    # Alpha jika butir dihapus, hanya untuk dimensi > 2 butir
    if len(daftar) > 2:
        for i, kolom in enumerate(daftar, 1):
            a_del = cronbach_alpha(x.drop(columns=kolom))
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(a_del, 3),
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)

print("\n=== TABEL RELIABILITAS ===")
display(tabel_reliabilitas)

print("\n=== ALPHA JIKA BUTIR DIHAPUS ===")
display(tabel_alpha_dihapus)



=== TABEL RELIABILITAS ===


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.550,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.678,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,Baik
3,Prokrastinasi akademik.,4,0.865,Baik
4,Respon emosional,5,0.555,Tidak reliabel



=== ALPHA JIKA BUTIR DIHAPUS ===


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.531
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.533
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.456
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.547
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.341
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.453
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.639
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.637
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.597
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.550


KeyboardInterrupt: 